# Gentamicin follow-up — seed noise floor and weight control

Two questions from the Site-D ablation, tested on Gentamicin (mirror:
`Drugs/Gentamicin/01-Run`, 40 rounds):

1. **Noise floor** — how large are per-site run-to-run differences under a
   *identical* configuration? Three seeds (42/43/44) × two arms
   (**withD**, **noD**), local runs with the same code path.
2. **Weight control** — is any effect of site D about its *data* or about its
   ~39 % share of the :term:`FedAvg` aggregation? The **weight** arm keeps all
   four clients but gives every client **uniform** aggregation weight
   (D keeps its data, loses its dominance).

Runs live in `Gentamicin/runs/{arm}_seed{seed}/results/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if not (HERE / "Gentamicin").exists():
    HERE = Path("/home/asd/Projects/Flower/DataSet/Dryad-DataSet/Analysis/08-Federated-mlp-lr-rf/Site-D-Ablation")
OUT = HERE / "comparison_results"; OUT.mkdir(exist_ok=True)

SITES = ["A", "B", "C", "D"]
ARMS = {"withD": [42, 43, 44], "noD": [42, 43, 44], "weight": [42, 43]}
METHODS = ["Centralized MLP", "Centralized RF", "FL FedAvg (MLP)", "FL FedAvg (LR)", "FL FedRF (Trees)"]

rows = []
for arm, seeds in ARMS.items():
    for seed in seeds:
        f = HERE / "Gentamicin" / "runs" / f"{arm}_seed{seed}" / "results" / "final_results.csv"
        if not f.exists():
            print("missing:", f); continue
        res = pd.read_csv(f)
        for m in METHODS:
            r = res[res["Method"] == m]
            if len(r) == 0:
                continue
            r = r.iloc[0]
            for s in SITES:
                rows.append({"arm": arm, "seed": seed, "method": m, "site": s,
                             "balacc": r[f"{s}_BalAcc"], "auc": r[f"{s}_AUC"]})
df = pd.DataFrame(rows)
df.to_csv(OUT / "followup_gent_per_site.csv", index=False)
print(f"loaded {len(df)} rows from {df.groupby('arm')['seed'].nunique().to_dict()} seeds per arm")


In [ ]:
# ── Noise floor: per-site mean ± SD across seeds (FedAvg MLP) ──
fav = df[df["method"] == "FL FedAvg (MLP)"]

piv = fav.pivot_table(index=["arm", "site"], values="balacc", aggfunc=["mean", "std"]).round(4)
print("FedAvg (MLP) — per-site Balanced Accuracy across seeds (mean / SD):")
display(piv)

print("\nWithin-arm noise (SD across seeds), per site:")
for arm in ARMS:
    sub = fav[fav["arm"] == arm]
    sds = sub.groupby("site")["balacc"].std().round(4)
    print(f"  {arm:7s}: {sds.to_dict()}")

# ── Paired deltas: noD − withD and weight − withD, per seed, all methods ──
def paired_deltas(method, a_arm, b_arm):
    rows = []
    sub = df[df["method"] == method]
    for seed in ARMS[a_arm]:
        if seed not in ARMS[b_arm]:
            continue
        a = sub[(sub["arm"] == a_arm) & (sub["seed"] == seed)].set_index("site")["balacc"]
        b = sub[(sub["arm"] == b_arm) & (sub["seed"] == seed)].set_index("site")["balacc"]
        if len(a) == 0 or len(b) == 0:
            continue
        for s in SITES:
            rows.append({"method": method, "contrast": f"{a_arm} - {b_arm}", "seed": seed,
                         "site": s, "delta": a[s] - b[s]})
    return pd.DataFrame(rows)

deltas = pd.concat([paired_deltas(m, a, b) for m in METHODS
                    for a, b in [("noD", "withD"), ("weight", "withD")]], ignore_index=True)
deltas.to_csv(OUT / "followup_gent_deltas.csv", index=False)
print("\nFedAvg (MLP) paired deltas (mean ± SD):")
display(deltas[deltas["method"] == "FL FedAvg (MLP)"]
        .pivot_table(index=["contrast", "site"], values="delta", aggfunc=["mean", "std"]).round(4))
print("Per-seed deltas (all methods):")
display(deltas.pivot_table(index="seed", columns=["method", "contrast", "site"], values="delta").round(3))


In [ ]:
# ── Figure 1: per-site Balanced Accuracy across seeds, three arms ──
arm_style = {"withD": ("#1f77b4", "with D (sample weights)"),
             "noD": ("#d62728", "no D"),
             "weight": ("#2ca02c", "uniform weights (D kept)")}
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True)
for ax, s in zip(axes, SITES):
    for xi, arm in enumerate(["withD", "noD", "weight"]):
        vals = fav[(fav["arm"] == arm) & (fav["site"] == s)]["balacc"].values
        color, _ = arm_style[arm]
        jitter = np.linspace(-0.07, 0.07, len(vals)) if len(vals) > 1 else [0.0]
        ax.scatter(np.full(len(vals), xi) + jitter, vals, color=color, s=26, zorder=3)
        ax.hlines(vals.mean(), xi - 0.28, xi + 0.28, color=color, lw=2.2, zorder=2)
    ax.set_xticks(range(3)); ax.set_xticklabels(["withD", "noD", "weight"], fontsize=9)
    ax.set_title(f"Site {s}", fontsize=10)
    ax.grid(True, axis="y", ls="--", lw=0.4, alpha=0.5)
    ax.set_ylim(0.35, 1.0)
axes[0].set_ylabel("FedAvg (MLP) Balanced Accuracy")
handles = [plt.Line2D([0], [0], color=c, lw=2.5, label=l) for c, l in arm_style.values()]
axes[0].legend(handles=handles, fontsize=7.5, loc="lower right")
fig.suptitle("Gentamicin — per-site Balanced Accuracy across seeds (dots = runs, lines = arm means)")
fig.tight_layout()
fig.savefig(OUT / "followup_gent_arms.pdf", bbox_inches="tight")
fig.savefig(OUT / "followup_gent_arms.svg", bbox_inches="tight")
plt.show()

# ── Figure 2: paired deltas vs zero (noise floor view) ──
fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.arange(len(SITES))
style = {"noD - withD": "#d62728", "weight - withD": "#2ca02c"}
for k, contrast in enumerate(style):
    sub = deltas[(deltas["method"] == "FL FedAvg (MLP)") & (deltas["contrast"] == contrast)]
    for xi, s in enumerate(SITES):
        vals = sub[sub["site"] == s]["delta"].values
        jitter = np.linspace(-0.06, 0.06, len(vals)) if len(vals) > 1 else [0.0]
        ax.scatter(np.full(len(vals), xi) + (k - 0.5) * 0.22 + jitter, vals,
                   color=style[contrast], s=26, zorder=3)
        mean, sd = vals.mean(), vals.std(ddof=1) if len(vals) > 1 else 0.0
        ax.errorbar(xi + (k - 0.5) * 0.22, mean, yerr=sd, color=style[contrast],
                    fmt="_", ms=18, capsize=4, lw=2, zorder=2)
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_xticks(x); ax.set_xticklabels([f"Site {s}" for s in SITES])
ax.set_ylabel("Δ Balanced Accuracy (no-D / uniform − with-D)")
handles = [plt.Line2D([0], [0], color=c, marker="_", ms=12, lw=2, label=l) for l, c in style.items()]
ax.legend(handles=handles, fontsize=8)
ax.grid(True, axis="y", ls="--", lw=0.4, alpha=0.5)
ax.set_title("Gentamicin — per-seed paired deltas (dots) with mean ± SD")
fig.tight_layout()
fig.savefig(OUT / "followup_gent_deltas.pdf", bbox_inches="tight")
fig.savefig(OUT / "followup_gent_deltas.svg", bbox_inches="tight")
plt.show()
